# Web Scraping - Seu Dinheiro


## Imports e configurações

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
import pathlib
from datetime import date

In [ ]:
base_url = "https://www.seudinheiro.com"
url_listagem = f"{base_url}/ultimas/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

## Diagnóstico



## Status pagina inicial

In [ ]:
resp_lista = requests.get(url_listagem, headers=headers, timeout=20)
print(f"Status HTTP: {resp_lista.status_code}")
print(f"URL final: {resp_lista.url}")
print(f"Tamanho do HTML: {len(resp_lista.text)} caracteres")

## 20 noticias por pag

In [ ]:
soup_lista = BeautifulSoup(resp_lista.text, "html.parser")

itens = soup_lista.find_all("div", class_="stream-item-container")
print(f"Itens de notícia encontrados na página 1: {len(itens)}")

## Uma noticia qualquer

In [ ]:
primeiro_item = soup_lista.find("div", class_="stream-item-container")

a_titulo = primeiro_item.find("h2", class_="feed_content_title").find("a")
link_teste = a_titulo["href"]
link_teste

## Noticia desbloqueada

In [ ]:
print(f"Testando notícia individual: {link_teste}")

resp_noticia = requests.get(link_teste, headers=headers, timeout=20)
print(f"Status HTTP da notícia: {resp_noticia.status_code}")

In [ ]:
soup_noticia = BeautifulSoup(resp_noticia.text, "html.parser")
soup_noticia

## Titulo

In [ ]:
titulo = soup_noticia.find("h1", class_="npsd_single-header-title")
print(titulo)
print()
print(titulo.get_text(strip=True))

## Subtitulo

In [ ]:
subtitulo = soup_noticia.find("p", class_="npsd_single-header-subtitle")
print(subtitulo)
print()
print(subtitulo.get_text(strip=True))

## Data e Hora

In [ ]:
data_hora = soup_noticia.find("div", class_="js-first-letter date-time")
print(data_hora)
print()
print(data_hora.get_text(strip=True))

In [ ]:
import re

texto_data_hora = data_hora.get_text(strip=True)

data = re.search(r"\d{1,2} de [a-zç]+ de \d{4}", texto_data_hora, flags=re.IGNORECASE)


print(data.group())
## Lembrar de tratar data


In [ ]:
from datetime import datetime

#texto_data = "20 de abril de 2026"
texto_data = data.group()

meses = {
    "janeiro": "01",
    "fevereiro": "02",
    "março": "03",
    "abril": "04",
    "maio": "05",
    "junho": "06",
    "julho": "07",
    "agosto": "08",
    "setembro": "09",
    "outubro": "10",
    "novembro": "11",
    "dezembro": "12"
}

partes = texto_data.lower().split(" de ")
dia = partes[0]
mes = meses[partes[1]]
ano = partes[2]

data_formatada = f"{ano}-{mes}-{dia.zfill(2)}"

print(data_formatada)

## Primeiras impressoes de um possivel db

In [ ]:
import pandas as pd

banco_noticias = {
    "titulo": titulo.get_text(' ',strip=True),
    "subtitulo": subtitulo.get_text(" ", strip=True),
    "data_texto": data_formatada,
}

print(banco_noticias)

## Adicionando informacoes do autor

In [ ]:
autor = soup_noticia.find("a", class_="npsd_single-main-content-author-info-content-title default")
print(autor.get_text(' ',strip=True))

In [ ]:
descricao_autor = soup_noticia.find("p", class_="author-bio-text")

descricao_autor_texto = descricao_autor.get_text(" ", strip=True) if descricao_autor else None
descricao_autor_texto

In [ ]:
banco_noticias = {
    "titulo": titulo.get_text(strip=True), ## Tem um \ax0 no texto do titulo, verificar 
    "subtitulo": subtitulo.get_text(" ", strip=True),
    "data_texto": data_formatada,
    'autor': autor.get_text(strip=True),
    'descricao_autor': descricao_autor_texto ## Existem descricoes que sao nulas
}
banco_noticias

## Informacoes do proprio texto

### Pegar caixa principal de noticias e limpar o que é propaganda, depois pegar só paragrafos (CONTINUA DEPOIS DA PUBLICIDADE, SAIBA MAIS)

In [ ]:
corpo = soup_noticia.find("div", class_="npsd_single-main-content-notice")
print(corpo)

In [ ]:
classes_lixo = [
     "borderAD",
    "ad-inview",
    "social-share",
    "social-share-mobile",
    "npsd_single-main-content-readtoo",
    "npsd_single-main-content-readtoo-article",
    "npsd_single-footer-author",
    "content-tags",
    "npsd_single-footer-banner",
    "banner",
    "banner-afterContent",
    "banner-afterContent--desktop"
]

for classe in classes_lixo:
    for tag in corpo.find_all(class_=classe):
        tag.decompose()

blocos = corpo.find_all('p')

lista_textos = []

for bloco in blocos:
    texto = bloco.get_text(" ", strip=True)

    if not texto:
        continue

    if texto.startswith("CONFIRA:"):
        continue

    if texto.startswith("SAIBA MAIS:"):
        continue

    lista_textos.append(texto)

texto_noticia = " ".join(lista_textos)

print(texto_noticia)


In [ ]:
banco_noticias = {
    "titulo": titulo.get_text(strip=True), ## Tem um \ax0 no texto do titulo, verificar 
    "subtitulo": subtitulo.get_text(" ", strip=True),
    "data_texto": data_formatada,
    'autor': autor.get_text(strip=True),
    'descricao_autor': descricao_autor_texto,
    'texto': texto_noticia 
}
banco_noticias

### Aqui temos toda a dinamica necessaria para montar o banco de dados, agora devemos iterar para todas as noticias

## Estrutura de repeticao


### Funcoes Necessarias

#### Limpar Texto

In [ ]:
import re
from datetime import datetime
import requests
from bs4 import BeautifulSoup

def limpar_texto(texto):
    if texto is None:
        return None
    
    texto = texto.replace("\xa0", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()



#### Converter Data Hora

In [ ]:
def converter_data_hora(texto_data_hora):
    data_texto = None
    hora_publicacao = None
    hora_atualizacao = None
    data_iso = None

    if texto_data_hora is None:
        return data_texto, hora_publicacao, hora_atualizacao, data_iso

    texto_data_hora = limpar_texto(texto_data_hora)

    data_match = re.search(
        r"\d{1,2} de [a-zç]+ de \d{4}",
        texto_data_hora,
        flags=re.IGNORECASE
    )

    horas = re.findall(
        r"\d{1,2}:\d{2}",
        texto_data_hora
    )

    if data_match:
        data_texto = data_match.group()

        meses = {
            "janeiro": "01",
            "fevereiro": "02",
            "março": "03",
            "marco": "03",
            "abril": "04",
            "maio": "05",
            "junho": "06",
            "julho": "07",
            "agosto": "08",
            "setembro": "09",
            "outubro": "10",
            "novembro": "11",
            "dezembro": "12"
        }

        partes = data_texto.lower().split(" de ")

        if len(partes) == 3:
            dia = partes[0].strip()
            mes_nome = partes[1].strip()
            ano = partes[2].strip()

            mes = meses.get(mes_nome)

            if mes is not None:
                data_iso = f"{ano}-{mes}-{dia.zfill(2)}"

    if len(horas) >= 1:
        hora_publicacao = horas[0]

    if len(horas) >= 2:
        hora_atualizacao = horas[1]

    return data_texto, hora_publicacao, hora_atualizacao, data_iso

#### Extrair Links de Paginas

In [ ]:

# def extrair_links_pagina(url_pagina, headers):
#     resp = requests.get(url_pagina, headers=headers, timeout=20)
#     soup = BeautifulSoup(resp.text, "html.parser")

#     itens = soup.find_all("div", class_="stream-item-container")

#     links = []

#     for item in itens:
#         h2 = item.find("h2", class_="feed_content_title")
#         if h2:
#             a_tag = h2.find("a")
#             if a_tag and a_tag.has_attr("href"):
#                 links.append(a_tag["href"])

#     return links

In [ ]:
from urllib.parse import urljoin

def extrair_links_pagina(url_pagina, headers, incluir_patrocinado=False):
    resp = requests.get(url_pagina, headers=headers, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    links = []

    # -----------------------------
    # Layout antigo
    # -----------------------------
    itens_antigos = soup.find_all("div", class_="stream-item-container")

    for item in itens_antigos:
        h2 = item.find("h2", class_="feed_content_title")

        if h2:
            a_tag = h2.find("a", href=True)

            if a_tag:
                link = urljoin(url_pagina, a_tag["href"])
                links.append(link)

    # -----------------------------
    # Layout novo
    # -----------------------------
    itens_novos = soup.find_all("div", class_="npsd_feed-ultimas-item")

    for item in itens_novos:
        h2 = item.find("h2", class_="npsd_feed-ultimas-item-content-title")

        if h2:
            a_tag = h2.find_parent("a", href=True)

            if a_tag:
                link = urljoin(url_pagina, a_tag["href"])
                links.append(link)

    # -----------------------------
    # Limpeza dos links
    # -----------------------------
    links_limpos = []

    for link in links:
        if not incluir_patrocinado and "/patrocinado/" in link:
            continue

        if link not in links_limpos:
            links_limpos.append(link)

    return links_limpos

#### Extrair infos Noticias Unicas

In [ ]:
def extrair_noticia_session(link, headers, session):
    resp_noticia = session.get(link, headers=headers, timeout=20)

    if resp_noticia.status_code != 200:
        raise Exception(f"HTTP {resp_noticia.status_code}")

    soup_noticia = BeautifulSoup(resp_noticia.text, "html.parser")

    # ----------------------------------------------------
    # Título
    # Layout novo: h1.npsd_single-header-title
    # Layout antigo: h1.newSingle_header_title
    # Fallback: meta og:title
    # ----------------------------------------------------
    titulo_tag = (
        soup_noticia.select_one("h1.npsd_single-header-title")
        or soup_noticia.select_one("h1.newSingle_header_title")
    )

    if titulo_tag:
        titulo_texto = limpar_texto(titulo_tag.get_text(" ", strip=True))
    else:
        meta_titulo = soup_noticia.find("meta", property="og:title")
        titulo_texto = limpar_texto(meta_titulo.get("content")) if meta_titulo else None

    # ----------------------------------------------------
    # Subtítulo
    # Layout novo: p.npsd_single-header-subtitle
    # Layout antigo: div.newSingle_header_excerpt
    # Fallback: meta description / og:description
    # ----------------------------------------------------
    subtitulo_tag = (
        soup_noticia.select_one("p.npsd_single-header-subtitle")
        or soup_noticia.select_one("div.newSingle_header_excerpt")
    )

    if subtitulo_tag:
        subtitulo_texto = limpar_texto(subtitulo_tag.get_text(" ", strip=True))
    else:
        meta_subtitulo = (
            soup_noticia.find("meta", property="og:description")
            or soup_noticia.find("meta", attrs={"name": "description"})
        )
        subtitulo_texto = limpar_texto(meta_subtitulo.get("content")) if meta_subtitulo else None

    # ----------------------------------------------------
    # Data e hora
    # Layout novo: div.js-first-letter.date-time
    # Layout antigo: div.single__date-time
    # ----------------------------------------------------
    data_hora_tag = (
        soup_noticia.select_one("div.js-first-letter.date-time")
        or soup_noticia.select_one("div.single__date-time")
    )

    texto_data_hora = (
        limpar_texto(data_hora_tag.get_text(" ", strip=True))
        if data_hora_tag
        else None
    )

    # Considerando a função converter_data_hora atualizada
    # retornando: data_texto, hora_publicacao, hora_atualizacao, data_iso
    data_texto, hora_publicacao, hora_atualizacao, data_iso = converter_data_hora(texto_data_hora)

    # ----------------------------------------------------
    # Autor
    # Layout novo: a.npsd_single-main-content-author-info-content-title
    # Layout antigo: div.newSingle_author_info_content_title
    # Fallback: meta name="author"
    # ----------------------------------------------------
    autor_tag = (
        soup_noticia.select_one("a.npsd_single-main-content-author-info-content-title")
        or soup_noticia.select_one("div.newSingle_author_info_content_title")
    )

    if autor_tag:
        autor_texto = limpar_texto(autor_tag.get_text(" ", strip=True))
    else:
        meta_autor = soup_noticia.find("meta", attrs={"name": "author"})
        autor_texto = limpar_texto(meta_autor.get("content")) if meta_autor else None

    # ----------------------------------------------------
    # Descrição do autor
    # Layout novo: p.author-bio-text
    # Layout antigo: p.author-bio__text
    # ----------------------------------------------------
    descricao_autor_tag = (
        soup_noticia.select_one("p.author-bio-text")
        or soup_noticia.select_one("p.author-bio__text")
    )

    descricao_autor_texto = (
        limpar_texto(descricao_autor_tag.get_text(" ", strip=True))
        if descricao_autor_tag
        else None
    )

    # ----------------------------------------------------
    # Corpo da notícia
    # Layout novo: div.npsd_single-main-content-notice
    # Layout antigo: div.newSingle_content_right
    # ----------------------------------------------------
    corpo = (
        soup_noticia.select_one("div.npsd_single-main-content-notice")
        or soup_noticia.select_one("div.newSingle_content_right")
    )

    texto_noticia = None

    if corpo:
        classes_lixo = [
            # anúncios
            "borderAD",
            "ad-inview",

            # compartilhamento
            "social-share",
            "social-share-mobile",
            "newSingle-shareMobile",

            # leia também / recomendações
            "npsd_single-main-content-readtoo",
            "npsd_single-main-content-readtoo-article",
            "newSingle_readtoo",
            "newSingle_readtoo_article",

            # autor / bio / rodapé da notícia
            "npsd_single-footer-author",
            "single__footer-author",

            # tags e banners finais
            "content-tags",
            "npsd_single-footer-banner",
            "banner",
            "banner-afterContent",
            "banner-afterContent--desktop",

            # layout antigo
            "single__first-letter",
        ]

        for classe in classes_lixo:
            for tag in corpo.find_all(class_=classe):
                tag.decompose()

        # Se quiser apenas parágrafos, use: corpo.find_all("p")
        # Se quiser manter subtítulos internos da notícia, use ["p", "h2"]
        blocos = corpo.find_all(["p", "h2"])

        lista_textos = []

        for bloco in blocos:
            texto = limpar_texto(bloco.get_text(" ", strip=True))

            if not texto:
                continue

            textos_descartar = [
                "CONFIRA:",
                "SAIBA MAIS:",
                "Leia também:",
                "LEIA TAMBÉM:",
                "CONTINUA DEPOIS DA PUBLICIDADE",
                "Compartilhar"
            ]

            if any(texto.startswith(prefixo) for prefixo in textos_descartar):
                continue

            lista_textos.append(texto)

        texto_noticia = "\n\n".join(lista_textos)

    return {
        "url": link,
        "titulo": titulo_texto,
        "subtitulo": subtitulo_texto,
        "data_texto": data_texto,
        "hora_publicacao": hora_publicacao,
        "hora_atualizacao": hora_atualizacao,
        "data_iso": data_iso,
        "autor": autor_texto,
        "descricao_autor": descricao_autor_texto,
        "texto": texto_noticia
    }

#### Extrair Infos noticia individual paralelizando

In [ ]:
import re
import time
import random
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
def worker_extrair_noticia(item, headers):
    link = item["url"]
    pagina = item["pagina_origem"]

    try:
        with requests.Session() as session:
            dados = extrair_noticia_session(link, headers, session)

        dados["pagina_origem"] = pagina
        dados["erro"] = None

    except Exception as e:
        dados = {
            "url": link,
            "pagina_origem": pagina,
            "titulo": None,
            "subtitulo": None,
            "data_texto": None,
            "hora_publicacao": None,
            "hora_atualizacao": None,
            "data_iso": None,
            "autor": None,
            "descricao_autor": None,
            "texto": None,
            "erro": str(e)
        }

    return dados

In [ ]:
base_url = "https://www.seudinheiro.com"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# 1) coletar links das 3 páginas
todos_links = []

for pagina in range(1, 1601):
    if pagina == 1:
        url_pagina = f"{base_url}/ultimas/"
    else:
        url_pagina = f"{base_url}/ultimas/pagina/{pagina}/"

    print(f"\nColetando links da página {pagina}: {url_pagina}")

    try:
        links = extrair_links_pagina(url_pagina, headers)
        print(f"Links encontrados: {len(links)}")

        for link in links:
            todos_links.append({
                "url": link,
                "pagina_origem": pagina
            })

        time.sleep(random.uniform(1, 2))

    except Exception as e:
        print(f"Erro ao coletar links da página {pagina}: {e}")

print(f"\nTotal de links coletados: {len(todos_links)}")

In [ ]:
registros = []

max_workers = 4


print(f"\nIniciando extração paralela com {max_workers} threads...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futuros = [
        executor.submit(worker_extrair_noticia, item, headers)
        for item in todos_links
    ]

    for i, futuro in enumerate(as_completed(futuros), start=1):
        try:
            dados = futuro.result()
            registros.append(dados)

            print(f"[{i}/{len(futuros)}] OK")
            print(f"  Página: {dados['pagina_origem']}")
            print(f"  Título: {dados['titulo']}")
            print(f"  Data: {dados.get('data_texto')} | Hora publicação: {dados.get('hora_publicacao')} | Hora atualização: {dados.get('hora_atualizacao')}")
            print(f"  Autor: {dados['autor']}")
            print(f"  Texto tamanho: {len(dados['texto']) if dados['texto'] else 0}")

        except Exception as e:
            print(f"[{i}/{len(futuros)}] ERRO: {e}")

In [ ]:
df_noticias = pd.DataFrame(registros)

print(f"\nTotal de notícias extraídas com sucesso: {len(df_noticias)}")
df_noticias.head(3)

In [ ]:
df_noticias = pd.DataFrame(registros)
print(df_noticias.shape)
df_noticias.head()
# df_noticias.to_csv(r"noticias_seu_dinheiro_paralelo_pag301_600.csv" ,index=False)

In [ ]:
# df_noticias.to_csv(
#     "noticias_seu_dinheiro_paralelo_pag1_27k.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

In [ ]:
# import polars as pl
# df_noticias = pl.scan_csv("noticias_seu_dinheiro_paralelo_pag1_27k.csv")
# df_noticias.sink_parquet(r'noticias_seu_dinheiro_paralelo_pag1_27k_v2_parquet.parquet')
# df_noticias = pl.scan_parquet(r'noticias_seu_dinheiro_paralelo_pag1_27k_v2_parquet.parquet')



# Amostrando base de dados

In [ ]:

# df_amostra = df_noticias.collect().sample(
#     n=10000,
#     seed=42
# )
# df_amostra.write_parquet(r'amostra_noticias_v1.parquet')



## Análises dos dados

In [ ]:
import polars as pl
df_amostra = pl.scan_parquet(r'amostra_noticias_v1.parquet')


In [ ]:
import polars as pl
import matplotlib.pyplot as plt

df_noticias = df_amostra.with_columns(
        pl.col("data_iso")
        .str.to_date("%Y-%m-%d")
        .alias("data_iso")
    ).with_columns(
        pl.col("data_iso")
        .dt.strftime("%Y-%m")
        .alias("ano_mes")
    )



noticias_por_mes = (
    df_noticias
    .filter(pl.col("data_iso").is_not_null())
    .group_by("ano_mes")
    .len(name="quantidade")
    .sort("ano_mes")
)

noticias_por_mes = noticias_por_mes.collect()


## Volumetria mensal

In [ ]:

plt.figure(figsize=(14, 5))

plt.bar(
    noticias_por_mes["ano_mes"].to_list(),
    noticias_por_mes["quantidade"].to_list()
)

plt.xlabel("Mês")
plt.ylabel("Número de notícias")
plt.title("Volumetria mensal de notícias")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Caracterizando autores e tentando encontrar alguma analise

In [ ]:
dim_autores = (
    df_noticias
    .select("autor")
    .filter(pl.col("autor").is_not_null())
    .unique()
    .sort("autor")
    .with_row_index(name="id_autor", offset=1)
)

dim_autores.select('autor').count().collect()

In [ ]:
df_noticias_com_id = (
    df_noticias
    .join(dim_autores, on="autor", how="left")
)

df_noticias_com_id.head().collect()#

In [ ]:
analise_autores = (
    df_noticias_com_id
    .group_by(["id_autor",'autor'])
    .agg([
        pl.len().alias("qtd_noticias"),
        pl.col("data_iso").min().alias("primeira_publicacao"),
        pl.col("data_iso").max().alias("ultima_publicacao"),
        pl.col("ano_mes").n_unique().alias("qtd_meses_com_publicacao"),
        pl.col("texto").str.len_chars().mean().alias("tamanho_medio_texto")
    ])
    .with_columns([
        (
            pl.col("ultima_publicacao") - pl.col("primeira_publicacao")
        ).dt.total_days().alias("dias_entre_primeira_e_ultima")
    ])
    .sort("qtd_noticias", descending=True)
)


analise_autores.head(55).collect().to_pandas()
